# Lab 0 — Meet the data: HumanEval and AeroEval
**Trusting AI-Written Code · Session 1 · before Lab 1** · about 25 minutes · no API key

Labs 1 and 2 grade AI-written code against two datasets. Before judging any score, you need to know what a problem looks like, what the model is shown, and what decides pass or fail. This lab opens both files, walks through them with tables and pictures, and **asks Colab's free AI model one problem from each** so you see a real prompt, a real answer and a real grade:

- **Part A — HumanEval** (a 10-problem sample): the public benchmark behind many headline scores.
- **Part B — AeroEval** (`aero20_s1.json`): the course's aerospace problem pack.

**Files:** upload `humaneval_sample.json` and `aero20_s1.json` next to this notebook (Files panel on the left), then Run all.

**Step 1 — Setup.** pandas and matplotlib are already in Colab.

In [ ]:
import json, math, os, textwrap, hashlib, urllib.request
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})
print("ready")

**Step 2 — Config.** Which files to open, and which model to ask (Colab's free model by default; no key).

In [ ]:
HE_FILE = "humaneval_sample.json"   # 10 HumanEval problems (5 of them are used in Lab 1)
PACK_FILE = "aero20_s1.json"         # the AeroEval pack for Labs 1-2; use "aero20.json" for the full 20-item pack
BASE_URL = ""                        # optional: a web folder holding both files
VERIFY_ONLINE = True                 # compare the HumanEval sample with the official copy, if the internet is reachable
DOMAIN = "aero"                      # course edition
MODEL = "colab:"                     # Colab's free, built-in Gemini (no key); "colab:<name>" picks a specific model
USE_CACHE = False                    # Lab 0 asks the model live - your answer may differ from your neighbour's

**Step 3 — Load both files** (helpers collapsed; open them if you're curious).

In [ ]:
# @title Loaders and display helpers { display-mode: "form" }
def _load(name):
    if not os.path.exists(name) and BASE_URL:
        try:
            urllib.request.urlretrieve(BASE_URL + name, name)
        except Exception:
            pass
    if not os.path.exists(name):
        raise FileNotFoundError(f"{name} not found: upload it next to this notebook (Files panel) or set BASE_URL in Step 2.")
    return json.load(open(name))

HE = _load(HE_FILE); PACK = _load(PACK_FILE)
HE_BY_ID = {r["task_id"]: r for r in HE["problems"]}
PROBLEMS = {p["id"]: p for p in PACK["problems"]}

def show_code(title, code, lang="python"):
    display(Markdown(f"**{title}**\n```{lang}\n{code.rstrip()}\n```"))

def short(x, n=60):
    s = x if isinstance(x, str) else json.dumps(x)
    s = s.replace("\n", " \u21b5 ")
    return s if len(s) <= n else s[:n - 1] + "\u2026"

def type_name(v):
    return {dict: "object", list: "list", str: "text", bool: "true/false", int: "number", float: "number"}.get(type(v), type(v).__name__)

def he_run(record, body):
    """HumanEval grading: prompt + completion + test, then check(entry_point). Passing = no assertion fails."""
    ns = {}
    try:
        exec(record["prompt"] + body + "\n" + record["test"] + f"\ncheck({record['entry_point']})\n", ns)
        return "pass", ""
    except AssertionError:
        return "FAIL", "an assertion failed"
    except Exception as e:
        return "FAIL", f"{type(e).__name__}: {e}"

def ref_function(code, entry):
    """The pack's OWN reference or planted code (trusted teaching code, not AI output)."""
    ns = {}; exec(PACK["imports"] + code, ns); return ns[entry]

def close(v, e, rel, ab=0.0):
    if isinstance(e, list):
        return isinstance(v, (list, tuple)) and len(v) == len(e) and all(close(a, b, rel, ab) for a, b in zip(v, e))
    return abs(v - e) <= max(rel * abs(e), ab)

def fmt(v):
    if isinstance(v, (list, tuple)):
        return "[" + ", ".join(fmt(x) for x in v) + "]"
    return f"{v:.6g}" if isinstance(v, float) else str(v)

def draw_tree(pack):
    """The pack's JSON structure as three columns: top level -> one problem -> one test case (and one defect)."""
    prob, test, dfc = pack["problems"][0], pack["problems"][0]["tests"][0], pack["defects"][0]
    fig, ax = plt.subplots(figsize=(12, 6.2)); ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")
    def column(x, title, d, highlight=(), w=0.21):
        keys = list(d); ys = [0.90 - i * min(0.068, 0.84 / max(len(keys), 1)) for i in range(len(keys))]
        ax.text(x, 0.97, title, ha="center", fontsize=9, weight="bold")
        pos = {}
        for k, y in zip(keys, ys):
            v = d[k]; size = f" [{len(v)}]" if isinstance(v, (list, dict)) else ""
            label = f"{k}: {type_name(v)}{size}" + ("" if isinstance(v, (list, dict)) else f"  = {short(v, 18)}")
            ax.text(x, y, label, ha="center", va="center", fontsize=7.2, family="monospace",
                    bbox=dict(boxstyle="round,pad=0.25", fc="#fde9c9" if k in highlight else "#f4f6f8", ec="#8a94a0", lw=0.5))
            pos[k] = y
        return pos
    top = column(0.13, f"{PACK_FILE}  (top level)", pack, highlight=("problems", "defects"), w=0.24)
    mid = column(0.44, f"problems[0]  ({prob['id']})", prob, highlight=("tests", "holdout_tests"))
    rt = column(0.76, "tests[0]  (one test case)", test)
    ax.text(0.76, 0.52, f"defects[0]  ({dfc['id']})", ha="center", fontsize=9, weight="bold")
    for i, k in enumerate(dfc):
        ax.text(0.76, 0.46 - i * 0.068, f"{k}: {short(dfc[k], 22)}", ha="center", va="center", fontsize=7.2, family="monospace",
                bbox=dict(boxstyle="round,pad=0.25", fc="#f8e1e1", ec="#8a94a0", lw=0.5))
    arrow = dict(arrowstyle="->", color="#1d6fb8", lw=1.0)
    ax.annotate("", xy=(0.335, 0.95), xytext=(0.245, top["problems"]), arrowprops=arrow)
    ax.annotate("", xy=(0.66, 0.95), xytext=(0.545, mid["tests"]), arrowprops=arrow)
    ax.annotate("", xy=(0.66, 0.52), xytext=(0.245, top["defects"]), arrowprops=dict(arrowstyle="->", color="#c0392b", lw=1.0))
    ax.set_title("Structure of the AeroEval pack: each arrow opens one item of a list", fontsize=10)
    plt.show()

def show_item(item_id):
    """Everything about one AeroEval problem, in tables."""
    p = PROBLEMS[item_id]; f = ref_function(p["canonical_solution"], p["entry_point"])
    display(Markdown(f"### {p['id']} \u2014 {p['area']} \u2014 `{p['entry_point']}`\n**Minimal prompt:** {p['prompt_minimal']}  \n**Gotcha:** {p['gotcha']}  \n**Source:** {p['source']}"))
    rows = [{"kind": kind, "inputs": fmt(t["args"]), "expected": fmt(t["expected"]), "rel_tol": t["rel_tol"], "abs_tol": t["abs_tol"],
             "reference result": fmt(f(*t["args"])), "matches": close(f(*t["args"]), t["expected"], t["rel_tol"], t["abs_tol"])}
            for kind, ts in (("visible", p["tests"]), ("held-out", p["holdout_tests"])) for t in ts]
    display(pd.DataFrame(rows))
    display(pd.DataFrame([{"sanity check": c["name"], "code": c["code"]} for c in p["sanity_checks"]]))

CACHE = {}

WRAPPER = "Write the Python function below. Reply with the code only, in one ```python block."
def make_prompt(p, variant):
    """The exact prompt sent: the same one-line WRAPPER, then the task text for the variant."""
    text = p["prompt_minimal"] if variant == "minimal" else p["prompt"]
    if variant == "tests_first":
        text += "\nIt must pass these tests:\n" + "\n".join(
            f"- {p['entry_point']}({', '.join(map(repr, t['args']))}) returns {t['expected']!r} (relative tolerance {t['rel_tol']:g}"
            + (f", absolute tolerance {t['abs_tol']:g}" if t.get("abs_tol") else "") + ")"
            for t in p["tests"])
    return WRAPPER + "\n\n" + text

def he_prompt(r):
    return "Complete this Python function. Reply with the whole function in one ```python block.\n\n" + r["prompt"]

def run_general(code, r, timeout=10):
    """HumanEval: the AI's function runs remotely; the problem's own check() runs HERE."""
    imports = "\n".join(l for l in r["prompt"].splitlines() if l.startswith(("from ", "import ")))
    try:
        with RemoteFunction(imports + "\n" + code, r["entry_point"], timeout) as f:
            g = {}
            exec(r["test"], g)
            g["check"](f)
        return True, None
    except AssertionError:
        return False, "a test assertion failed"
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"[:120]

def ask_free_model(prompt, key):
    """One live answer from the model in Step 2 (Colab's free Gemini by default). None if it is unavailable."""
    try:
        reply = ask_llm(prompt, key)[0]
        print(f"live answer from {MODEL or 'colab:'} ({'Colab default model' if MODEL in ('', 'colab:') else 'as set in Step 2'})")
        return reply
    except Exception as e:
        print(f"The model is not available here ({type(e).__name__}). Run this notebook in Google Colab, signed in, "
              "with MODEL = \"colab:\" - the rest of the lab works without it.")
        return None

def show_ae_result(p, res):
    """The model's answers next to the expected values: visible tests, held-out cases, sanity checks."""
    rows = [{"kind": kind, "inputs": fmt(t["args"]), "expected": fmt(t["expected"]),
             "model's code returned": fmt(x.get("value")) if x.get("value") is not None else "(error)", "pass": x["ok"]}
            for kind, ts, xs in (("visible", p["tests"], res["tests"]), ("held-out", p["holdout_tests"], res["holdout"])) for t, x in zip(ts, xs)]
    rows += [{"kind": "sanity check", "inputs": c["name"], "expected": "holds", "model's code returned": "holds" if ok else "fails", "pass": ok}
             for c, ok in zip(p["sanity_checks"], res["checks"] or [None] * len(p["sanity_checks"]))]
    display(pd.DataFrame(rows))
    if res.get("error"):
        print("error:", res["error"])
    print("Verdict:", "visible tests PASS" if res["passed"] else "visible tests FAIL", "|",
          "held-out PASS" if res["holdout_passed"] else "held-out FAIL", "|", "sanity checks PASS" if res["checks_passed"] else "sanity checks FAIL")

def he_table():
    rows = [{"task_id": r["task_id"], "function": r["entry_point"], "prompt lines": r["prompt"].count("\n"),
             "solution lines": r["canonical_solution"].strip().count("\n") + 1, "asserts in test": r["test"].count("assert "),
             "helper in prompt": r["prompt"].count("def ") > 1, "random test": "random" in r["test"],
             "in Lab 1": r["used_in_lab1"], "why it is here": r["why_selected"]} for r in HE["problems"]]
    t = pd.DataFrame(rows); display(t)
    fig, ax = plt.subplots(figsize=(9, 3))
    ax.bar(t["task_id"].str.replace("HumanEval/", "HE/"), t["asserts in test"], color=["#1d6fb8" if x else "#9aa5b1" for x in t["in Lab 1"]])
    for i, v in enumerate(t["asserts in test"]):
        ax.text(i, v + 0.1, str(v), ha="center", fontsize=8)
    ax.set_ylabel("assert statements"); ax.set_title("Test strength varies a lot (blue = used in Lab 1)"); plt.show()
    print("A loop counts as one assert: HE/32 and HE/53 run theirs 100 times, on random inputs (HE/53 unseeded).")

HE_FIELDS = ["task_id", "prompt", "entry_point", "canonical_solution", "test"]
def verify_official():
    """Compare each sample record's fingerprint with the official Hugging Face copy (needs internet)."""
    sha = lambda r: hashlib.sha256(json.dumps({k: r[k] for k in HE_FIELDS}, sort_keys=True).encode()).hexdigest()
    try:
        from datasets import load_dataset
        official = {r["task_id"]: dict(r) for r in load_dataset("openai/openai_humaneval", split="test")}
        same = {r["task_id"]: sha(official[r["task_id"]]) == r["record_sha256"] for r in HE["problems"]}
        print(f"Official check: {sum(same.values())}/{len(same)} records identical to the official dataset",
              "" if all(same.values()) else f"- differ: {[k for k, v in same.items() if not v]}")
    except Exception as e:
        print(f"Official copy not reachable here ({type(e).__name__}); the sample's own tests still pass above.")

ABOUT = {"pack": "name", "version": "pack version", "domain": "edition", "lens": "label for the engineering notes",
         "grading_policy": "how the tests are meant to be read", "imports": "imports every solution runs with",
         "generated_by": "the script that built and verified it", "generated_at": "build time",
         "problems": "the problems", "defects": "planted wrong versions, for testing the tests",
         "canonical_content_sha256": "fingerprint of the content"}
def pack_top_level():
    display(pd.DataFrame([{"key": k, "type": type_name(v), "size": len(v) if isinstance(v, (list, dict)) else "",
                           "what it is": ABOUT.get(k, ""), "value": "" if k in ("problems", "defects") else short(v, 50)}
                          for k, v in PACK.items()]))

def problems_view():
    rows = []
    for p in PACK["problems"]:
        f = ref_function(p["canonical_solution"], p["entry_point"])
        cases = p["tests"] + p["holdout_tests"]
        rows.append({"id": p["id"], "area": p["area"], "function": p["entry_point"], "visible tests": len(p["tests"]),
                     "held-out cases": len(p["holdout_tests"]), "sanity checks": len(p["sanity_checks"]),
                     "rel_tol": p["tests"][0]["rel_tol"],
                     "reference reproduces all cases": all(close(f(*t["args"]), t["expected"], t["rel_tol"], t["abs_tol"]) for t in cases),
                     "the classic slip": p["gotcha"]})
    t = pd.DataFrame(rows); display(t)
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw={"width_ratios": [1, 2]})
    t["area"].value_counts().sort_values().plot.barh(ax=a1, color="#1d6fb8"); a1.set_title("Problems per area"); a1.set_xlabel("problems")
    t.set_index("id")[["visible tests", "held-out cases", "sanity checks"]].plot.bar(ax=a2, stacked=True, color=["#1d6fb8", "#f0a830", "#7fb77e"])
    a2.set_title("Checks per problem"); a2.set_ylabel("count"); a2.tick_params(axis="x", rotation=0)
    plt.tight_layout(); plt.show()
    print(f"Reference solutions reproduce {sum(t['reference reproduces all cases'])}/{len(t)} problems' visible and held-out values.")

def prompt_lengths():
    lens = pd.DataFrame([{"id": p["id"], "minimal": len(p["prompt_minimal"]), "full": len(p["prompt"])} for p in PACK["problems"]]).set_index("id")
    ax = lens.plot.bar(figsize=(9, 2.8), color=["#9aa5b1", "#1d6fb8"]); ax.set_ylabel("characters"); ax.tick_params(axis="x", rotation=0)
    ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), frameon=False)
    ax.set_title("How much each prompt specifies: minimal vs full"); plt.tight_layout(); plt.show()

def defects_view():
    D = pd.DataFrame([{"defect": d["id"], "problem": d["problem"], "what is wrong": d["description"],
                       "test": d["expected_detection"]["test"], "sanity checks": d["expected_detection"]["sanity"]} for d in PACK["defects"]])
    display(D)
    grid = (D[["test", "sanity checks"]] == "caught").astype(int).values
    fig, ax = plt.subplots(figsize=(4.2, 0.42 * len(D) + 0.9)); from matplotlib.colors import ListedColormap
    ax.imshow(grid, cmap=ListedColormap(["#f6cfcb", "#cfe8d2"]), vmin=0, vmax=1, aspect="auto")
    ax.set_xticks([0, 1], ["test", "sanity checks"]); ax.set_yticks(range(len(D)), D["defect"] + "  " + D["problem"])
    for i in range(len(D)):
        for j in range(2):
            ax.text(j, i, "caught" if grid[i, j] else "MISSED", ha="center", va="center", fontsize=8)
    ax.set_title("Who catches each planted defect"); plt.tight_layout(); plt.show()

def plot_d1():
    p = PROBLEMS["AE01"]; good = ref_function(p["canonical_solution"], p["entry_point"])
    bad = ref_function(next(d for d in PACK["defects"] if d["id"] == "D1")["code"], p["entry_point"])
    h = [i * 100.0 for i in range(111)]
    fig, ax = plt.subplots(figsize=(8, 3.4))
    ax.plot(h, [good(x)[1] / 1000 for x in h], label="reference", color="#1d6fb8")
    ax.plot(h, [bad(x)[1] / 1000 for x in h], "--", label="D1 (planted)", color="#c0392b")
    for t in p["tests"]:
        ax.plot(t["args"][0], t["expected"][1] / 1000, "o", color="#1d6fb8")
    for t in p["holdout_tests"]:
        ax.plot(t["args"][0], t["expected"][1] / 1000, "s", color="#f0a830")
    ax.set_xlabel("geopotential altitude (m)"); ax.set_ylabel("pressure (kPa)"); ax.legend(title="circles: visible tests; squares: held-out")
    ax.set_title("AE01: both curves fall with altitude; only reference values expose D1"); plt.show()
    print(f"At 5,000 m: reference {good(5000.0)[1]:,.1f} Pa, D1 {bad(5000.0)[1]:,.1f} Pa (that is the pressure at about 5 m)")

print(f"HumanEval sample: {len(HE['problems'])} problems | {PACK['pack']} v{PACK['version']}: "
      f"{len(PACK['problems'])} problems, {len(PACK['defects'])} planted defects")

The course's grader (collapsed): the same code Labs 1 and 2 use. The AI's code runs in its own process; every pass/fail decision is made here in the notebook. It is fault containment, not a security sandbox.

In [ ]:
# @title Helpers — run this cell (no need to read it) { display-mode: "form" }
# Standard helpers for every lab notebook in the course. AI-written code runs in its own
# process and only computes outputs; every pass/fail decision is made in this notebook.
# Fault containment, NOT a security sandbox: use a clean runtime (no secrets, no mounted Drive). Identical in every notebook,
# inlined (not imported) so each notebook stands alone. Domain-free: the pack carries
# the domain. Relies on PACK, CACHE, USE_CACHE, MODEL from the config cell.
import json, re, sys, math, hashlib, subprocess, tempfile, textwrap, datetime, platform

def prompt_sha(text):
    return hashlib.sha256(text.encode()).hexdigest()[:12]

def ask_llm(prompt, key, n=1, temperature=None):
    """Return n completions. Cached by default (deterministic class runs); live otherwise."""
    if USE_CACHE:
        entry = CACHE.get(key)
        if entry is None or not entry.get("samples"):
            raise KeyError(f"no cached response for {key}" + (f" from CACHE_MODEL={globals().get('CACHE_MODEL')!r}" if globals().get("CACHE_MODEL") else "")
                           + "; set USE_CACHE = False to call a model")
        want = prompt_sha(prompt)
        matched = [s for s in entry["samples"] if s.get("prompt_sha") == want]
        samples = matched or entry["samples"]
        entry_sha = entry.get("prompt_sha")
        sample_shas = {s.get("prompt_sha") for s in samples if s.get("prompt_sha")}
        if not matched and ((sample_shas and want not in sample_shas) or (entry_sha and entry_sha != want)):
            print(f"note: prompt for {key} changed since it was cached - this response is NOT counted as evidence")
        return [s["text"] for s in samples[:n]]
    temperature = globals().get("TEMPERATURE", 0.8) if temperature is None else temperature
    out = []
    for _ in range(n):
        if MODEL.startswith("colab:"):                      # free in Colab, no key
            from google.colab import ai                     # verify the API each term
            name = MODEL.split(":", 1)[1] or None
            try:
                out.append(ai.generate_text(prompt, model_name=name) if name else ai.generate_text(prompt))
            except TypeError:
                out.append(ai.generate_text(prompt))
        else:                                               # any provider via LiteLLM + your key
            import litellm
            r = litellm.completion(model=MODEL, temperature=temperature,
                                   messages=[{"role": "user", "content": prompt}])
            out.append(r.choices[0].message.content)
    return out

def provenance(key, prompt=None):
    """Where a response came from: 'fixture' (test data, never evidence), 'capture:<model>', or 'live:<model>'.
    Adds '(prompt changed)' if the cached response was generated from a different prompt."""
    if not USE_CACHE:
        return f"live:{MODEL}"
    entry = CACHE.get(key, {}); samples = entry.get("samples") or [{}]
    want = prompt_sha(prompt) if prompt is not None else None
    chosen = next((s for s in samples if s.get("prompt_sha") == want), samples[0]) if want else samples[0]
    m = chosen.get("model", "unknown")
    tag = "fixture" if m.startswith("fixture:") else f"capture:{m}"
    if want is not None:
        own = chosen.get("prompt_sha", entry.get("prompt_sha"))
        if own and own != want:
            tag += " (prompt changed)"
    return tag

def extract_code(text):
    """First fenced python block, else the whole text."""
    m = re.search(r"```(?:python)?\s*\n(.*?)```", text, re.S)
    return (m.group(1) if m else text).strip() + "\n"

# ---------------------------------------------------------------- grading: trust boundary
# The AI's code runs in its OWN process and only computes outputs for inputs it is sent.
# Every comparison, sanity check and pass/fail decision is made here, in the notebook, so the
# AI's code cannot fake a printed result or patch the grader's arithmetic. Values come back as
# plain JSON over a private pipe (not stdout) and are parsed safely. This is fault containment,
# NOT a security sandbox: the child can still read files (including the pack's reference values),
# inspect its parent via the OS, or use the network - so use a clean runtime with no secrets.
import os as _os, time as _time, selectors as _selectors, secrets as _secrets, signal as _signal
for _m in ("pandas", "csv", "hypothesis"):            # import before any AI code runs (no file shadowing)
    try:
        __import__(_m)
    except ImportError:
        pass

_CHILD = r"""
import sys, os, json, math
rfd, wfd, code_path, entry = int(sys.argv[1]), int(sys.argv[2]), sys.argv[3], sys.argv[4]
rin, wout = os.fdopen(rfd, "r"), os.fdopen(wfd, "w")
def enc(v):
    if hasattr(v, "tolist"): v = v.tolist()
    elif hasattr(v, "item") and not isinstance(v, (str, bytes)): v = v.item()
    if isinstance(v, bool) or v is None or isinstance(v, (int, str)): return v
    if isinstance(v, float): return v if math.isfinite(v) else {"__float__": repr(v)}
    if isinstance(v, complex): return {"__complex__": [v.real, v.imag]}
    if isinstance(v, tuple): return {"__tuple__": [enc(x) for x in v]}
    if isinstance(v, (set, frozenset)): return {"__set__": [enc(x) for x in v]}
    if isinstance(v, list): return [enc(x) for x in v]
    if isinstance(v, dict): return {"__dict__": [[enc(k), enc(x)] for k, x in v.items()]}
    return {"__repr__": repr(v)[:200]}
def dec(v):
    if isinstance(v, list): return [dec(x) for x in v]
    if isinstance(v, dict):
        if "__tuple__" in v: return tuple(dec(x) for x in v["__tuple__"])
        if "__dict__" in v: return {dec(k): dec(x) for k, x in v["__dict__"]}
    return v
def send(obj):
    wout.write(json.dumps(obj) + "\n"); wout.flush()
ns = {}
try:
    exec(compile(open(code_path).read(), "<candidate>", "exec"), ns)
    f = ns[entry]
    send({"ready": True})
except BaseException as e:
    send({"load_error": f"{type(e).__name__}: {e}"[:200]}); sys.exit(0)
for line in rin:
    req = json.loads(line)
    try:
        send({"id": req["id"], "value": enc(f(*dec(req["args"]), **dec(req.get("kwargs", {}))))})
    except BaseException as e:
        send({"id": req["id"], "error": f"{type(e).__name__}: {e}"[:200]})
"""

class CandidateError(Exception):
    """The AI-written code raised, failed to load, broke the protocol, or timed out."""

def _dec(v):
    if isinstance(v, list):
        return [_dec(x) for x in v]
    if isinstance(v, dict):
        if "__tuple__" in v: return tuple(_dec(x) for x in v["__tuple__"])
        if "__set__" in v: return set(_dec(x) for x in v["__set__"])
        if "__float__" in v: return float(v["__float__"]) if v["__float__"] in ("inf", "-inf", "nan") else None
        if "__complex__" in v: return complex(*v["__complex__"])
        if "__dict__" in v: return {_dec(k): _dec(x) for k, x in v["__dict__"]}
        return ("<unrecognised value>", str(v)[:80])
    return v

def _enc(v):
    if isinstance(v, tuple): return {"__tuple__": [_enc(x) for x in v]}
    if isinstance(v, list): return [_enc(x) for x in v]
    if isinstance(v, dict): return {"__dict__": [[_enc(k), _enc(x)] for k, x in v.items()]}
    return v

class RemoteFunction:
    """Calls the AI-written function inside a separate process. Use as a context manager."""
    def __init__(self, code, entry, timeout=10):
        self.deadline = _time.monotonic() + timeout
        self.tmp = tempfile.mkdtemp(prefix="cand_")
        self.p = self.err = None
        self.w1 = self.r2 = None
        self.sel = _selectors.DefaultSelector()
        self.buf = b""
        self._closed = False
        try:
            code_path, child_path = _os.path.join(self.tmp, "candidate.py"), _os.path.join(self.tmp, "child.py")
            open(code_path, "w").write(code); open(child_path, "w").write(_CHILD)
            r1, self.w1 = _os.pipe(); self.r2, w2 = _os.pipe()
            env = {"PATH": "/usr/local/bin:/usr/bin:/bin", "HOME": self.tmp, "LANG": "C.UTF-8"}   # reduced, not isolation
            self.err = open(_os.path.join(self.tmp, "stderr.txt"), "w")
            self.p = subprocess.Popen([sys.executable, "-I", child_path, str(r1), str(w2), code_path, entry],
                                      pass_fds=(r1, w2), stdin=subprocess.DEVNULL, stdout=subprocess.DEVNULL,
                                      stderr=self.err, env=env, cwd=self.tmp, start_new_session=True)
            _os.close(r1); _os.close(w2)
            self.sel.register(self.r2, _selectors.EVENT_READ)
            msg = self._read()
            if "load_error" in msg:
                raise CandidateError(msg["load_error"])
            if not msg.get("ready"):
                raise CandidateError("protocol violation at start-up")
        except BaseException:
            self.close()        # kill the child, close fds, delete the scratch dir, on ANY failure
            raise

    def _read(self):
        while b"\n" not in self.buf:
            left = self.deadline - _time.monotonic()
            if left <= 0 or not self.sel.select(timeout=left):
                raise CandidateError("timeout")
            chunk = _os.read(self.r2, 65536)
            if chunk:
                self.buf += chunk
                if len(self.buf) > 4_000_000:            # 4 MB cap on one response line
                    raise CandidateError("protocol violation (oversized response)")
                continue
            if not chunk:
                tail = open(self.err.name).read().strip().splitlines()
                raise CandidateError("process ended" + (f": {tail[-1][:120]}" if tail else ""))
        line, self.buf = self.buf.split(b"\n", 1)
        try:
            return json.loads(line)
        except ValueError:
            raise CandidateError("protocol violation")
    def __call__(self, *args, **kwargs):
        rid = _secrets.token_hex(8)
        try:
            _os.write(self.w1, (json.dumps({"id": rid, "args": _enc(list(args)), "kwargs": _enc(kwargs)}) + "\n").encode())
        except OSError:
            raise CandidateError("process ended")
        msg = self._read()
        if msg.get("id") != rid:
            raise CandidateError("protocol violation")
        if "error" in msg:
            raise CandidateError(msg["error"])
        return _dec(msg.get("value"))
    def close(self):
        if self._closed:
            return
        self._closed = True
        if self.p is not None:
            for _ in range(50):                      # SIGKILL can't interrupt a syscall mid-import; wait briefly
                try:
                    _os.killpg(self.p.pid, _signal.SIGKILL)
                except (ProcessLookupError, PermissionError, OSError):
                    break
                try:
                    if self.p.wait(timeout=0.1) is not None:
                        break
                except subprocess.TimeoutExpired:
                    continue
                except Exception:
                    break
        try:
            self.sel.close()
        except Exception:
            pass
        for fd in (self.w1, self.r2):
            if fd is not None:
                try:
                    _os.close(fd)
                except OSError:
                    pass
        if self.err is not None:
            try:
                self.err.close()
            except Exception:
                pass
        import shutil as _shutil
        _shutil.rmtree(self.tmp, ignore_errors=True)       # remove the scratch directory

    def __enter__(self):
        return self
    def __exit__(self, *exc):
        self.close()

_MAX_ABS = 1e308          # anything bigger (or a huge int) is treated as an invalid return, never compared

def _close_enough(v, e, rel_tol, abs_tol=0.0):
    """Decided here, with this process's own arithmetic. Rejects wrong TYPES, non-finite and
    oversized values rather than raising, so a hostile return cannot crash the notebook."""
    if isinstance(e, list):
        return isinstance(v, (list, tuple)) and len(v) == len(e) and all(_close_enough(a, b, rel_tol, abs_tol) for a, b in zip(v, e))
    if isinstance(v, bool) or not isinstance(v, (int, float)):
        return False
    try:
        if isinstance(v, int) and abs(v) > _MAX_ABS:       # avoid int->float OverflowError
            return False
        if not math.isfinite(v) or abs(v) > _MAX_ABS:
            return False
        return abs(v - e) <= max(rel_tol * abs(e), abs_tol)
    except (OverflowError, ValueError, ArithmeticError):
        return False

def _run_tests(f, tests):
    out = []
    for t in tests:
        try:
            v = f(*t["args"])
            out.append({"ok": _close_enough(v, t["expected"], t["rel_tol"], t.get("abs_tol", 0.0)), "value": v})
        except CandidateError as e:
            if str(e).startswith(("timeout", "process ended", "protocol")):
                raise
            out.append({"ok": False, "value": None, "error": str(e)[:120]})
    return out

def evaluate(code, problem, timeout=10):
    """Grade AI-written code: it runs in its own process; tests, held-out cases and sanity checks
    are evaluated HERE. Pass a problem copy with "tests": [] (checks only) or "sanity_checks": []
    (tests only)."""
    r = {"tests": [], "holdout": [], "checks": [], "error": None}
    try:
        with RemoteFunction(PACK.get("imports", "") + "\n" + code, problem["entry_point"], timeout) as f:
            r["tests"] = _run_tests(f, problem["tests"])
            r["holdout"] = _run_tests(f, problem.get("holdout_tests", []))
            for c in problem.get("sanity_checks", []):
                try:
                    exec(c["code"], {"f": f, "math": math})
                    r["checks"].append(True)
                except CandidateError as e:
                    if str(e).startswith(("timeout", "process ended", "protocol")):
                        raise
                    r["checks"].append(False)
                except Exception:
                    r["checks"].append(False)
    except CandidateError as e:
        r["error"] = str(e)[:160]
        r["tests"] = [dict(t, ok=False) for t in r["tests"]] or []
        r["checks"] = [False] * len(problem.get("sanity_checks", [])) if problem.get("sanity_checks") else []
    r["passed"] = bool(r["tests"]) and all(t["ok"] for t in r["tests"]) and r["error"] is None
    r["holdout_passed"] = (all(t["ok"] for t in r["holdout"]) and r["error"] is None) if r["holdout"] else None
    r["checks_passed"] = (all(r["checks"]) and r["error"] is None) if r["checks"] else None
    return r

def is_evidence(source):
    """Only real captures and live calls count as model evidence; fixtures and mismatched captures never do."""
    return str(source).startswith(("capture:", "live:")) and "(prompt changed)" not in str(source)


def wilson(k, n, z=1.96):
    """95% Wilson interval for k successes in n trials."""
    if n == 0:
        return (0.0, 1.0)
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

def show_grid(rows, cols, cell):
    """Coloured pass/fail grid in Colab (pandas Styler); plain table elsewhere."""
    import pandas as pd
    df = pd.DataFrame([[cell(r, c) for c in cols] for r in rows], index=rows, columns=cols)
    try:
        from IPython.display import display
        display(df.style.map(lambda v: "background:#cfe9dd" if v == "pass" else
                             "background:#f2cccc" if v == "FAIL" else ""))
    except Exception:
        print(df.to_string())
    return df

def write_artifacts(results, extra=None):
    """A1 results.csv + A2 run_manifest.json — every lab writes both."""
    import csv
    keys = sorted({k for r in results for k in r})
    with open("results.csv", "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=keys); w.writeheader(); w.writerows(results)
    manifest = {"pack": PACK.get("pack"), "pack_version": PACK.get("version"),
                "pack_canonical_content_sha256": PACK.get("canonical_content_sha256", PACK.get("sha256")),
                "pack_file_sha256": globals().get("PACK_FILE_SHA256"),
                "requested_temperature": globals().get("TEMPERATURE"),
                "temperature_applied": ("cached samples: " + ", ".join(sorted({str(x.get("temperature") if x.get("temperature") is not None else "provider default") for v in CACHE.values() for x in v.get("samples", [])})))
                                       if USE_CACHE else ("provider default (not controllable)" if str(MODEL).startswith("colab:") else globals().get("TEMPERATURE")),
                "domain": DOMAIN, "model": MODEL, "use_cache": USE_CACHE,
                "cache_models": sorted({s.get("model", "?") for e in CACHE.values() for s in e.get("samples", [])}),
                "run_date": datetime.date.today().isoformat(), "python": platform.python_version()}
    manifest.update(extra or {})
    json.dump(manifest, open("run_manifest.json", "w"), indent=1)
    print("wrote results.csv and run_manifest.json")
    return manifest

print("helpers ready")


---
## Part A — HumanEval
**Step 4 — One HumanEval record, and the ten sample problems.** Every problem has the same five fields. The model is given only the **prompt** and writes the body; the canonical solution and the test stay hidden from it. Five of the ten are the ones Lab 1 uses.

In [ ]:
r = HE_BY_ID["HumanEval/0"]
display(pd.DataFrame([{"field": k, "type": type_name(r[k]), "characters": len(r[k]), "what it is": HE["fields"][k],
                       "start of value": short(r[k])} for k in HE_FIELDS]))
show_code("prompt - what the model is given", r["prompt"])
show_code("test - check(candidate) decides pass or fail (hidden)", r["test"])
he_table()

**Step 5 — How HumanEval decides pass or fail.** Glue prompt + body + test together and call `check(entry_point)`: no failed assert means pass. Each canonical solution is graded the same way; then the sample is compared with the official dataset (optional; needs internet).

In [ ]:
graded = {r["task_id"]: he_run(r, r["canonical_solution"])[0] for r in HE["problems"]}
display(pd.DataFrame([graded]))
print("Exact equality (==) for most answers; a tolerance only where the test author wrote one.")
if VERIFY_ONLINE:
    verify_official()

**Aha — a hard-coded answer passes a one-assert test.** *This wrong code is instructor-injected, not a model's output.* HumanEval/34 asks for the sorted unique elements of any list. This 'solution' ignores its input and returns the one answer the test checks.

In [ ]:
r, fake = HE_BY_ID["HumanEval/34"], "    return [0, 2, 3, 5, 9, 123]\n"
show_code("instructor-injected wrong body", fake)
print("HumanEval's own test says:", he_run(r, fake)[0])
ns = {}; exec(r["prompt"] + fake, ns)
print("but unique([3, 1, 3]) returns", ns["unique"]([3, 1, 3]), "instead of [1, 3]")

A benchmark score is only as good as its tests. EvalPlus (HumanEval+) added many tests per problem and found solutions that passed the originals but were wrong.

**Step 6 — Ask the free model a HumanEval problem.** *This is a live answer from Colab's free model, not a recording: run it again and it may differ.* You see the exact prompt sent, the raw reply, the code taken from it, and HumanEval's own verdict. The model's code runs in its own process.

In [ ]:
r = HE_BY_ID["HumanEval/0"]; prompt = he_prompt(r)
show_code("1. the exact prompt sent", prompt, "text")
reply = ask_free_model(prompt, "HumanEval/0|lab0")
if reply:
    show_code("2. the raw reply, unedited", reply, "text")
    code = extract_code(reply); ok, why = run_general(code, r)
    show_code("3. the code taken from the reply", code)
    print("4. HumanEval's own check():", "PASS" if ok else f"FAIL - {why}")

Passing here means the model's function survived **this problem's seven asserts** — no more. Lab 1 asks: on how many problems, with what uncertainty, and against how strong a test?

---
## Part B — AeroEval
**Step 7 — The pack: structure and problems.** One JSON object; two lists do the work, `problems` and `defects`. Visible tests can appear in a prompt (Lab 2's tests-first prompt shows them); held-out cases never do. They are withheld from the prompt, not secret: they sit in this file. The table re-runs every reference solution on every case.

In [ ]:
pack_top_level()
draw_tree(PACK)
problems_view()

**Step 8 — Anatomy of one problem: AE16, stall speed.** Every field in one place, then how much the full prompt specifies compared with the minimal one (Lab 1 compares them).

In [ ]:
show_item("AE16")
show_code("full prompt - the contract: units, constants, assumptions", PROBLEMS["AE16"]["prompt"])
show_code("canonical_solution - the reference implementation", PROBLEMS["AE16"]["canonical_solution"])
prompt_lengths()

**Step 9 — Ask the free model an AeroEval problem.** *A live answer from Colab's free model; run it again and it may differ.* AE16 with its **full** prompt. The model's numbers are compared with the pack's visible tests, held-out cases (never shown to the model) and sanity checks — the same grading Labs 1 and 2 use.

In [ ]:
p = PROBLEMS["AE16"]; prompt = make_prompt(p, "full")
show_code("1. the exact prompt sent", prompt, "text")
reply = ask_free_model(prompt, "AE16|full|lab0")
if reply:
    show_code("2. the raw reply, unedited", reply, "text")
    code = extract_code(reply); show_code("3. the code taken from the reply", code)
    show_ae_result(p, evaluate(code, p))

**Step 10 — Planted defects and what catches them.** *Each defect is an instructor-planted wrong version, never a model's output and never scored.* The pack records whether the reference **test** and the **sanity checks** catch it.

In [ ]:
defects_view()

**Aha — see a planted defect.** *D1 is instructor-planted.* It converts altitude to kilometres but keeps the lapse rate per metre, so 5,000 m is computed as 5 m. Pressure still falls with height, so a 'pressure decreases' check passes, but the scale is badly wrong.

In [ ]:
plot_d1()

**Step 11 — HumanEval and AeroEval side by side.**

| | HumanEval | AeroEval |
|---|---|---|
| What a problem is | a general programming puzzle | an aerospace formula or solver, with units |
| Pass or fail decided by | `check(candidate)`: assert statements | reference values with a stated tolerance |
| Comparison | mostly exact (`==`) | relative and absolute tolerance per test |
| Units and conventions | rarely matter | the main source of silent errors |
| Extra checks | none | sanity checks (physical properties) per problem |
| Held-out cases | none | two per problem, never in a prompt |
| Planted defects | none | yes, to test the tests |
| Public? | yes, widely (models may have seen it) | course-built (less likely to be memorised) |

**Check your understanding.**

1. In HumanEval, what does the model see, and what decides pass or fail?
2. Why can a wrong answer pass HumanEval/34?
3. In AeroEval, what is the difference between a visible test and a held-out case, and is a held-out case secret?
4. Why does the D1 curve fool a 'pressure falls with height' check?
5. In Steps 6 and 9, what exactly did a PASS establish, and what did it not?

<details><summary>Answers</summary>

1. Only the prompt (signature + docstring). The hidden `check(candidate)` asserts decide.
2. Its test checks a single input, so returning that one answer passes.
3. A visible test may be shown in a prompt; a held-out case never is. It is withheld from the prompt, not secret: it sits in this file.
4. A consistent units error keeps the curve falling; only a reference value at a known altitude shows the scale is wrong.
5. That one answer, on one run, matched these particular tests. Not that the model is reliable: a second run can differ, and one problem says nothing about the rest.
</details>

## Try it on your own
Ask the free model any AeroEval problem, with the **minimal** or the **full** prompt, and compare. Pick an id from Step 7. Try AE01 or AE14 with the minimal prompt — do the units and conventions survive?

In [ ]:
ITEM, VARIANT = "AE14", "minimal"   # VARIANT: "minimal" or "full"
p = PROBLEMS[ITEM]; prompt = make_prompt(p, VARIANT)
show_code("the exact prompt sent", prompt, "text")
reply = ask_free_model(prompt, f"{ITEM}|{VARIANT}|lab0")
if reply:
    code = extract_code(reply); show_code("the code taken from the reply", code)
    show_ae_result(p, evaluate(code, p))

**Next: Lab 1.** You have seen what the model is shown, what decides pass or fail, and one live answer from each dataset. Lab 1 asks a model for every problem and measures what happens — with honest uncertainty.